# 1. Unit Testing with `unittest`

`unittest` is Python's built-in unit testing framework. It helps you verify that small pieces of code behave as expected and makes it easier to catch regressions when your code changes.

With `unittest`, you can build test cases, fixtures, suites, and test runners. This notebook introduces the most common pieces you need to start writing reliable automated tests.

In [1]:
class Calculator:
    def add(self, a, b):
        return a + b

    def divide(self, a, b):
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

In a regular project, tests usually live in a separate file such as `test_calculator.py`. In this notebook, we keep the examples inline so you can focus on the testing patterns first.

In [ ]:
import unittest

class TestCalculator(unittest.TestCase):
    def setUp(self):
        self.calc = Calculator()

    def test_add(self):
        self.assertEqual(self.calc.add(2, 3), 5)
        self.assertEqual(self.calc.add(-1, 1), 0)
        self.assertEqual(self.calc.add(0, 0), 0)

    def test_divide(self):
        self.assertEqual(self.calc.divide(10, 2), 5)
        self.assertAlmostEqual(self.calc.divide(5, 2), 2.5)

    def test_divide_by_zero(self):
        with self.assertRaises(ValueError):
            self.calc.divide(10, 0)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCalculator)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

...
----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


### What This Example Shows

- `setUp()` prepares a fresh test object before each test method runs
- `assertEqual()` checks exact expected values
- `assertAlmostEqual()` is useful when comparing floating-point results
- `assertRaises()` verifies that the correct exception is raised

When your tests are stored in a file, you can run them from the command line with:

`python -m unittest test_calculator.py`

You can also run narrower targets such as a specific class or a specific test method.

The **TestCase** class provides several assert methods to check for and report failures.

The following table lists the most commonly used methods (see the tables below for more assert methods):

### Common Assertion Methods

Some of the most useful assertion helpers are:

- `assertEqual(a, b)`
- `assertTrue(x)`
- `assertFalse(x)`
- `assertIn(item, container)`
- `assertAlmostEqual(a, b)`
- `assertRaises(ExceptionType)`

These methods make test failures easier to read than plain `if` statements.

## 2. Running Tests

You can run unittest from the command line at different levels:

`python -m unittest test_module1 test_module2`
`python -m unittest test_module.TestClass`
`python -m unittest test_module.TestClass.test_method`

These commands let you run multiple modules, one test class, or one individual test method.

### Core Concepts in `unittest`

The framework uses a few recurring ideas:

- **test fixture**: the setup state a test needs before it runs
- **test case**: one class containing related test methods
- **test suite**: a grouped collection of tests to run together
- **test runner**: the component that executes tests and reports results

Typical fixtures include temporary files, sample objects, or database setup used only during testing.

## 3. Organizing Test Code

A `unittest.TestCase` subclass groups related test methods. Each test method should check one behavior or one small scenario. In larger projects, this makes failures easier to interpret and maintain.

Two lifecycle methods are especially common:

- `setUp()`: runs before each test method
- `tearDown()`: runs after each test method

Use them when each test needs preparation or cleanup.

This working environment is called a **test fixture**. A fresh `TestCase` instance is created for each test method, which helps keep tests isolated from one another.

When you want to run a custom group of tests, you can build a `TestSuite`. In many simple modules, `unittest.main()` is enough, but suites are useful when you want finer control over which tests run together.

In [4]:
import unittest

class TestMath(unittest.TestCase):
    def test_add(self):
        self.assertEqual(1 + 1, 2)

    def test_subtract(self):
        self.assertEqual(5 - 3, 2)

class TestString(unittest.TestCase):
    def test_upper(self):
        self.assertEqual("foo".upper(), "FOO")

    def test_isupper(self):
        self.assertTrue("FOO".isupper())
        self.assertFalse("Foo".isupper())

def build_test_suite():
    suite = unittest.TestSuite()
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestMath))
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestString))
    return suite

runner = unittest.TextTestRunner()
runner.run(build_test_suite())

....
----------------------------------------------------------------------
Ran 4 tests in 0.004s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>

**When to Use Test Suites?**

*   When you want to group tests by feature, module, or priority.
*   To run a subset of tests instead of the entire test collection.
*   To organize tests logically for better maintainability.

## 3.1 Skipping Tests

`unittest` can skip individual tests, whole classes, or conditionally skip based on the environment. This is useful when a test depends on an operating system, a Python version, or a feature that is temporarily unavailable.

Common approaches include:

- `@unittest.skip(...)` for unconditional skips
- `@unittest.skipIf(...)` and `@unittest.skipUnless(...)` for conditional skips
- `self.skipTest(...)` inside a test or fixture when a runtime condition appears

In [ ]:
import unittest
import sys

# Skip the entire test class if Python version is less than 3.8
@unittest.skipIf(sys.version_info < (3, 8), "Requires Python 3.8 or higher")
class TestFeature(unittest.TestCase):

    def test_feature_a(self):
        self.assertEqual(2 + 2, 4)

    @unittest.skip("Temporarily skipping this test")
    def test_feature_b(self):
        self.fail("This test is skipped and should not run")

    @unittest.skipIf(sys.platform == "darwin", "Skip on macOS")
    def test_feature_c(self):
        self.assertTrue(True)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestFeature)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

.ss.ss.s.s.....
----------------------------------------------------------------------
Ran 15 tests in 0.017s

OK (skipped=6)


## 4. Parameterized-Style Tests

### 4.1 Subtests

When the same test logic should run against several inputs, `subTest()` is often the simplest built-in approach. It keeps the test compact while still reporting which input caused a failure.

In [ ]:
import unittest

class NumbersTest(unittest.TestCase):
    def test_even(self):
        for value in [0, 2, 4, 8, 10]:
            with self.subTest(value=value):
                self.assertEqual(value % 2, 0)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(NumbersTest)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

....
----------------------------------------------------------------------
Ran 4 tests in 0.003s

OK


In [ ]:
import unittest

class TestSquare(unittest.TestCase):
    def test_square(self):
        test_cases = [
            (2, 4),
            (3, 9),
            (4, 16),
        ]
        for value, expected in test_cases:
            with self.subTest(value=value, expected=expected):
                self.assertEqual(value * value, expected)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestSquare)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

.ss.FFss.s.s......
FAIL: test_even (__main__.NumbersTest.test_even) (i=5)
Test that numbers 0, 2, 4 are even.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipython-input-24-2145519258.py", line 10, in test_even
    self.assertEqual(i % 2, 0)
AssertionError: 1 != 0

FAIL: test_even (__main__.NumbersTest.test_even) (i=9)
Test that numbers 0, 2, 4 are even.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipython-input-24-2145519258.py", line 10, in test_even
    self.assertEqual(i % 2, 0)
AssertionError: 1 != 0

----------------------------------------------------------------------
Ran 17 tests in 0.015s

FAILED (failures=2, skipped=6)


### 4.2 Using the `parameterized` Package

If you want a more decorator-based style for multiple input cases, you can use the third-party `parameterized` package. This is optional; for many projects, `subTest()` is already enough.

In [ ]:
try:
    from parameterized import parameterized
except ModuleNotFoundError:
    parameterized = None
    print("Optional dependency not installed: python -m pip install parameterized")

import unittest

if parameterized is not None:
    class TestSquareParameterized(unittest.TestCase):
        @parameterized.expand([
            (2, 4),
            (3, 9),
            (4, 16),
        ])
        def test_square(self, value, expected):
            self.assertEqual(value * value, expected)

    suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestSquareParameterized)
    runner = unittest.TextTestRunner(verbosity=2)
    runner.run(suite)

Optional dependency not installed: python -m pip install parameterized
